# 1 — Setup & Data Exploration

Install dependencies, download VQA v2.0, and explore the data.

**Run this notebook once** to populate the `data/` directory. After that, go straight to notebook 2.

In [ ]:
!pip install -q torch torchvision transformers matplotlib tqdm Pillow h5py

In [ ]:
import json
import os
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Download VQA v2.0

Downloads questions and annotations (~50 MB). Add `download_images=True` to also fetch MS-COCO images (~19 GB).

In [ ]:
QUESTIONS_URLS = {
    "train": "https://s3.amazonaws.com/cvqa/v2_Questions_Train_mscoco.zip",
    "val":   "https://s3.amazonaws.com/cvqa/v2_Questions_Val_mscoco.zip",
}
ANNOTATIONS_URLS = {
    "train": "https://s3.amazonaws.com/cvqa/v2_Annotations_Train_mscoco.zip",
    "val":   "https://s3.amazonaws.com/cvqa/v2_Annotations_Val_mscoco.zip",
}
IMAGES_URLS = {
    "train": "http://images.cocodataset.org/zips/train2014.zip",
    "val":   "http://images.cocodataset.org/zips/val2014.zip",
}


def download_file(url, dest):
    """Download url to dest with a progress bar. Skips if file exists."""
    dest = Path(dest)
    if dest.exists():
        print(f"  [skip] {dest.name} already exists")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  Downloading {dest.name} ...")

    class _Bar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with _Bar(unit="B", unit_scale=True, desc=dest.name) as t:
        urllib.request.urlretrieve(url, dest, reporthook=t.update_to)


def download_and_extract(urls, dest_dir):
    """Download and extract all zips in urls dict."""
    for split, url in urls.items():
        filename = url.rsplit("/", 1)[-1]
        zip_path = dest_dir / filename
        download_file(url, zip_path)
        if zip_path.exists():
            print(f"  Extracting {zip_path.name} ...")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(dest_dir)
            zip_path.unlink()


def organise(data_dir):
    """Move extracted files into questions/, answers/, images/ subdirectories."""
    for d in ["questions", "answers", "images"]:
        (data_dir / d).mkdir(parents=True, exist_ok=True)

    for f in data_dir.glob("v2_OpenEnded_mscoco_*_questions.json"):
        f.rename(data_dir / "questions" / f.name)
    for f in data_dir.glob("v2_mscoco_*_annotations.json"):
        f.rename(data_dir / "answers" / f.name)
    for split in ("train2014", "val2014"):
        src = data_dir / split
        if src.exists():
            src.rename(data_dir / "images" / split)


def download_vqa(data_dir, download_images=False):
    """Download the full VQA v2.0 dataset."""
    data_dir = Path(data_dir)

    print("=== Questions ===")
    download_and_extract(QUESTIONS_URLS, data_dir)

    print("\n=== Annotations ===")
    download_and_extract(ANNOTATIONS_URLS, data_dir)

    if download_images:
        print("\n=== Images (this will take a while) ===")
        download_and_extract(IMAGES_URLS, data_dir)

    print("\n=== Organising ===")
    organise(data_dir)
    print("Done!")

In [ ]:
# Set download_images=True to also fetch ~19 GB of MS-COCO images
download_vqa(DATA_DIR, download_images=False)

## Inspect Questions and Annotations

In [ ]:
with open(DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json") as f:
    questions = json.load(f)["questions"]
with open(DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json") as f:
    annotations = json.load(f)["annotations"]

print(f"Questions:   {len(questions):,}")
print(f"Annotations: {len(annotations):,}")
print("\nSample question:", questions[0])
print("\nSample annotation:", annotations[0])

## Answer Distribution

In [ ]:
answer_counts = Counter(ann["multiple_choice_answer"] for ann in annotations)
top_20 = answer_counts.most_common(20)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([a for a, _ in top_20][::-1], [c for _, c in top_20][::-1])
ax.set_xlabel("Count")
ax.set_title("Top 20 Answers")
plt.tight_layout()
plt.show()

# Coverage analysis
total = len(annotations)
for k in [100, 500, 1000, 2000]:
    covered = sum(c for _, c in answer_counts.most_common(k))
    print(f"Top {k:>5d} answers cover {covered / total * 100:.1f}% of samples")

## Sample Image-Question-Answer Triplets

In [ ]:
ann_by_qid = {ann["question_id"]: ann for ann in annotations}
img_dir = DATA_DIR / "images" / "train2014"

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, q in zip(axes.flat, questions[:6]):
    img_id = q["image_id"]
    img_path = img_dir / f"COCO_train2014_{img_id:012d}.jpg"
    answer = ann_by_qid[q["question_id"]]["multiple_choice_answer"]
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    ax.set_title(f"Q: {q['question']}\nA: {answer}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

Data is ready. Proceed to **Notebook 2** for model training, evaluation, and visualization.

In [ ]:
# Verify directory structure
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.relative_to(DATA_DIR)}  ({size_mb:.1f} MB)")
    elif p.is_dir():
        n_files = len(list(p.iterdir()))
        print(f"  {p.relative_to(DATA_DIR)}/  ({n_files} items)")